# Train YOLOv8 on UA-DETRAC (Google Colab Pro) - interruption-resilient

Workflow:
1. Verify GPU runtime
2. Install dependencies
3. Mount Google Drive (where the raw DETRAC dataset lives)
4. Convert DETRAC -> YOLO format (run once)
5. **Train YOLOv8 with per-epoch Drive sync** (last.pt + best.pt copied to Drive after every epoch)
5b. **Resume from Drive checkpoint** (if Colab disconnected mid-training)
6. Evaluate on val split
7. Final copy of weights to Drive
8. Visualize predictions

**Resilience design.** Ultralytics writes `last.pt` after every epoch and `best.pt` whenever val mAP improves, but only on Colab's local disk. We add an `on_fit_epoch_end` callback that copies these files (plus `results.csv`) to Drive after every epoch, so an interrupted run loses at most one epoch.

**Before running**, in Colab go to `Runtime -> Change runtime type` and select:
- Hardware accelerator: **GPU**
- GPU type: **T4** (free) or **L4 / A100** (Colab Pro)
- Runtime shape: **High-RAM** (Pro)

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

Colab already has PyTorch + CUDA installed; we just need ultralytics.

In [ ]:
# Install ultralytics. The Colab base image periodically ships numpy 2.x while
# ultralytics' dependency chain still expects numpy 1.x in some sub-paths.
# pip will print 'dependency conflict' WARNINGS for unrelated packages
# (rasterio, jax, cupy, shap, ...) -- those are safe to ignore for YOLO training.
#
# After this cell finishes, run the import-check cell. If it FAILS, do
#   Runtime -> Restart session
# and re-run cells from the top (the install will be a no-op).
%pip install -q "ultralytics>=8.2,<9" "numpy<2" pyyaml pillow tqdm
print('Install done. If the next cell errors, do Runtime -> Restart session.')


In [ ]:
try:
    import numpy as np
    import torch
    import ultralytics
    print('numpy        ', np.__version__)
    print('torch        ', torch.__version__)
    print('cuda available', torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
    print('ultralytics  ', ultralytics.__version__)
    print()
    print('OK -- environment is ready.')
except Exception as e:
    print('IMPORT FAILED:', e)
    print()
    print('Fix: in Colab menu, click  Runtime -> Restart session,')
    print('then re-run the cells from the top (install will be a no-op).')


## 3. Mount Google Drive

Upload the DETRAC dataset (or a zip of it) to your Drive once, e.g.
`MyDrive/PFE/DETRAC.zip`.

Tip: upload a zipped copy and unzip into Colab's local disk (`/content/`) for
fast IO during training. Drive read is much slower than local SSD.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# adjust to where you placed the dataset on Drive
DRIVE_DETRAC = '/content/drive/MyDrive/PFE/DETRAC'
DRIVE_REPO   = '/content/drive/MyDrive/PFE/minimal_research'
DRIVE_RUNS   = '/content/drive/MyDrive/PFE/runs'

RUN_NAME      = 'yolov8n_detrac'
LOCAL_PROJECT = '/content/runs/detrac'
DRIVE_RUN_DIR = os.path.join(DRIVE_RUNS, RUN_NAME)

os.makedirs(DRIVE_RUNS, exist_ok=True)
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)
print('Drive paths configured.')
print('  local project   :', LOCAL_PROJECT)
print('  drive mirror    :', DRIVE_RUN_DIR)

### 3a. If DETRAC is uploaded as a zip, unzip to local Colab disk

In [ ]:
%%bash -e
if [ -f /content/drive/MyDrive/PFE/DETRAC.zip ]; then
    mkdir -p /content/DETRAC
    unzip -q -o /content/drive/MyDrive/PFE/DETRAC.zip -d /content/DETRAC
    ls /content/DETRAC | head
else
    echo 'No /content/drive/MyDrive/PFE/DETRAC.zip found; using DRIVE_DETRAC path directly.'
fi

In [ ]:
import os
DETRAC_ROOT = '/content/DETRAC' if os.path.isdir('/content/DETRAC/Insight-MVT_Annotation_Train') else DRIVE_DETRAC
print('Using DETRAC_ROOT =', DETRAC_ROOT)

## 4. Convert DETRAC -> YOLO format

Copy the converter from your Drive-mirrored repo to local disk so we can run it.

In [ ]:
import os, shutil
if not os.path.exists('/content/minimal_research'):
    if os.path.isdir(DRIVE_REPO):
        os.makedirs('/content/minimal_research/tools', exist_ok=True)
        shutil.copy(os.path.join(DRIVE_REPO, 'tools', 'detrac_to_yolo.py'),
                    '/content/minimal_research/tools/detrac_to_yolo.py')
        print('Copied converter from Drive.')
    else:
        print('Upload the repo to Drive at', DRIVE_REPO)

In [ ]:
%cd /content/minimal_research
!python tools/detrac_to_yolo.py \
    --detrac-root {DETRAC_ROOT} \
    --out         /content/dataset \
    --split       train \
    --val-frac    0.1 \
    --frame-stride 5

In [ ]:
!cat /content/dataset/data.yaml
!echo '---'
!ls /content/dataset/images/train | head
!echo '---'
!wc -l /content/dataset/labels/train/*.txt | tail -1

## 5. Train YOLOv8 (with per-epoch Drive sync)

Recipe:
- `yolov8n.pt` start weights -> transfer learning from COCO
- `imgsz=640`, `batch=32` (T4/L4) or `64` (A100)
- `epochs=80` with `patience=15` (early stop on val mAP plateau)
- `optimizer=SGD`, `lr0=0.01`, `cos_lr=True`
- mosaic on for first 70 epochs, off for last 10
- `degrees=0` (surveillance cameras are static)

**Resilience.** A custom `on_fit_epoch_end` callback copies `last.pt`,
`best.pt`, and `results.csv` to Drive after every epoch. If Colab
disconnects, the last-completed-epoch weights are already on Drive.

In [ ]:
# Drive-sync callback: runs after every training epoch (validation done).
import os, shutil, time

def make_drive_sync_callback(drive_run_dir: str):
    os.makedirs(drive_run_dir, exist_ok=True)
    os.makedirs(os.path.join(drive_run_dir, 'weights'), exist_ok=True)

    def _copy_safe(src, dst):
        if os.path.exists(src):
            try:
                shutil.copy(src, dst)
            except Exception as e:
                print('[drive-sync] failed to copy', src, ':', e)

    def on_fit_epoch_end(trainer):
        save_dir = str(trainer.save_dir)
        epoch = int(trainer.epoch) + 1
        t0 = time.time()
        _copy_safe(os.path.join(save_dir, 'weights', 'last.pt'),
                   os.path.join(drive_run_dir, 'weights', 'last.pt'))
        _copy_safe(os.path.join(save_dir, 'weights', 'best.pt'),
                   os.path.join(drive_run_dir, 'weights', 'best.pt'))
        _copy_safe(os.path.join(save_dir, 'results.csv'),
                   os.path.join(drive_run_dir, 'results.csv'))
        _copy_safe(os.path.join(save_dir, 'args.yaml'),
                   os.path.join(drive_run_dir, 'args.yaml'))
        print('[drive-sync] epoch', epoch, '->', drive_run_dir,
              '({:.1f}s)'.format(time.time() - t0))

    return on_fit_epoch_end

print('Sync callback ready. Drive run dir:', DRIVE_RUN_DIR)

In [ ]:
from ultralytics import YOLO

# fresh start from COCO-pretrained weights
model = YOLO('yolov8n.pt')
model.add_callback('on_fit_epoch_end', make_drive_sync_callback(DRIVE_RUN_DIR))

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=80,
    imgsz=640,
    batch=32,
    optimizer='SGD',
    lr0=0.01,
    cos_lr=True,
    patience=15,
    mosaic=1.0,
    close_mosaic=10,
    degrees=0.0,
    flipud=0.0,
    fliplr=0.5,
    project=LOCAL_PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    plots=True,
    save=True,
    save_period=10,
    verbose=True,
)

## 5b. Resume from Drive checkpoint (if Colab disconnected)

**Use this section ONLY if a previous run was interrupted.** Skip to section 6
for a fresh successful run.

What to do when Colab dies mid-training:
1. Reconnect / open a fresh runtime.
2. Re-run cells 1-4 (verify GPU, install ultralytics, mount Drive, set paths).
3. Re-run the conversion cell (fast) if `/content/dataset/` was lost.
4. Run the resume cells below.

Ultralytics' `resume=True` reloads optimizer state, learning-rate schedule,
and epoch counter from `last.pt`.

In [ ]:
# pull the most recent checkpoint back from Drive to local disk
import os, shutil
drive_last = os.path.join(DRIVE_RUN_DIR, 'weights', 'last.pt')
drive_best = os.path.join(DRIVE_RUN_DIR, 'weights', 'best.pt')
drive_args = os.path.join(DRIVE_RUN_DIR, 'args.yaml')

local_run = os.path.join(LOCAL_PROJECT, RUN_NAME)
os.makedirs(os.path.join(local_run, 'weights'), exist_ok=True)
if os.path.exists(drive_last):
    shutil.copy(drive_last, os.path.join(local_run, 'weights', 'last.pt'))
    print('Restored last.pt from Drive')
if os.path.exists(drive_best):
    shutil.copy(drive_best, os.path.join(local_run, 'weights', 'best.pt'))
    print('Restored best.pt from Drive')
if os.path.exists(drive_args):
    shutil.copy(drive_args, os.path.join(local_run, 'args.yaml'))
    print('Restored args.yaml from Drive')

ckpt = os.path.join(local_run, 'weights', 'last.pt')
if not os.path.exists(ckpt):
    print('No checkpoint found on Drive. Start fresh from cell 5 above.')
else:
    import torch
    sd = torch.load(ckpt, map_location='cpu')
    print('Checkpoint epoch:', sd.get('epoch', '?'))
    print('Best fitness so far:', sd.get('best_fitness', '?'))

In [ ]:
# resume training
from ultralytics import YOLO
ckpt = os.path.join(LOCAL_PROJECT, RUN_NAME, 'weights', 'last.pt')

model = YOLO(ckpt)
model.add_callback('on_fit_epoch_end', make_drive_sync_callback(DRIVE_RUN_DIR))
results = model.train(resume=True)

## 6. Evaluate on val

In [ ]:
best = os.path.join(LOCAL_PROJECT, RUN_NAME, 'weights', 'best.pt')
model = YOLO(best)
metrics = model.val(data='/content/dataset/data.yaml', imgsz=640, batch=32)
print('mAP50-95:', metrics.box.map)
print('mAP50   :', metrics.box.map50)
print('per-class mAP50:', dict(zip(['car', 'bus', 'van', 'others'], metrics.box.maps.tolist())))

## 7. Final copy of weights to Drive

The per-epoch sync already keeps Drive up to date; this final cell makes
sure plots and the very last `best.pt` are also synced.

In [ ]:
import shutil, os
src_run = os.path.join(LOCAL_PROJECT, RUN_NAME)
dst_dir = DRIVE_RUN_DIR
os.makedirs(os.path.join(dst_dir, 'weights'), exist_ok=True)

for fname in ['last.pt', 'best.pt']:
    src = os.path.join(src_run, 'weights', fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(dst_dir, 'weights', fname))

for fname in ['results.csv', 'args.yaml', 'results.png',
              'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    src = os.path.join(src_run, fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(dst_dir, fname))

print('Final sync to', dst_dir, '-- contents:')
for f in sorted(os.listdir(dst_dir)):
    print(' ', f)
for f in sorted(os.listdir(os.path.join(dst_dir, 'weights'))):
    print('  weights/', f)

## 8. Visualize a few predictions on val

In [ ]:
import glob, random
val_imgs = glob.glob('/content/dataset/images/val/*.jpg')
sample = random.sample(val_imgs, min(8, len(val_imgs)))
results = model(sample, imgsz=640, conf=0.25)
from IPython.display import Image, display
for r in results:
    out_path = r.save(filename='/content/preds.jpg')
    display(Image(out_path))

---

## What gets saved to Drive

After a successful (or interrupted) run, your `MyDrive/PFE/runs/yolov8n_detrac/`
folder contains:

```
yolov8n_detrac/
  weights/
    last.pt              # most recent epoch (use this to resume)
    best.pt              # best val mAP so far (use this for the attack)
  results.csv
  args.yaml
  results.png
  confusion_matrix.png
  PR_curve.png
  F1_curve.png
```

## Next step

Back on your local machine:

1. Download `best.pt` from `MyDrive/PFE/runs/yolov8n_detrac/weights/best.pt`
2. Place it at `D:/minimal_research/runs/yolov8n_detrac/best.pt`
3. Edit `configs/default.yaml`:
   ```yaml
   detector:
     weights: "runs/yolov8n_detrac/best.pt"
   ```
4. Drop a few DETRAC val frames into `data/images/` and run
   `python scripts/sanity_check.py --image data/images/<frame>.jpg`.